# NFL Coaching Career Movement Prediction

This notebook analyzes NFL coaching career trajectories and predicts whether a coach will be promoted, demoted, move sideways, or exit the league.

## ⚙️ v2 — Leakage-corrected version

Revised copy of `ML_project_final.ipynb`. Changes vs. the original:

1. **Group-aware train/test split.** The original used a random `train_test_split`. Because each coach has many seasons and labels are forward-looking, that leaked correlated rows across train/test and inflated scores. We now use `GroupShuffleSplit` on `CoachID` so no coach appears in both sets (Preprocessing cell and the 5-tier remap cell).
2. **Group-aware cross-validation.** `cross_val_score` and both `GridSearchCV`s now use `StratifiedGroupKFold` with `groups=CoachID`, so the CV estimate isn't leaky either.
3. **Bug fix:** the `min_samples_split` grid started at `1` (invalid for sklearn integers — would raise an error); it now starts at `2`.
4. **Bug fix:** the 5-tier remap now also refreshes `Next_Year` (it was stale after re-sorting), so the `Exit/Gap` gap-logic is correct.

**Expect lower — but honest — scores than the original.**

> ⚠️ Still-known limitations (intentionally *not* changed here, to keep this diff focused): `statistical_positionality` and `prev_league_churn` are engineered on the full dataset before the split (a milder form of leakage), and current-season (2025) rows are still labeled `Exit/Gap` due to right-censoring.


## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, cross_val_score, GridSearchCV,
                                     GroupShuffleSplit, StratifiedGroupKFold)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_selection import f_classif
from sklearn.inspection import permutation_importance

from imblearn.over_sampling import SMOTE
from collections import Counter

import nflreadpy as nfl
import warnings
warnings.filterwarnings('ignore')

print("All imports loaded successfully.")


## 2. Data Loading & Team Name Mapping

In [ ]:
# Load NFL team mapping
teams = nfl.load_teams()
mapping = dict(zip(teams['team_abbr'], teams['team_name']))

# Load coaching dataset and team stats
df = pd.read_stata("coach list 89-25 with race.dta")
df2 = pd.read_csv("team_stats_2003_2023.csv")

print(f"Coaching data: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Team stats: {df2.shape[0]} rows, {df2.shape[1]} columns")
df.head(3)


In [ ]:
df2.info()

## 3. Data Cleaning & Merging

In [ ]:
#Translate team abbreviations to full names, accounting for franchise relocations
def get_historical_name(row):
    translations = {
        'GNB': 'GB', 'KAN': 'KC', 'NEW': 'NE', 
        'NOR': 'NO', 'SFO': 'SF', 'TAM': 'TB'
    }
    abbr = translations.get(row['Team'], row['Team'])
    year = row['Year']
    
    if abbr == 'WAS':
        if year <= 2019: return 'Washington Redskins'
        if year <= 2021: return 'Washington Football Team'
        return 'Washington Commanders'
    if abbr in ['STL', 'LA', 'LAR']:
        return 'St. Louis Rams' if year <= 2015 else 'Los Angeles Rams'
    if abbr in ['SD', 'LAC']:
        return 'San Diego Chargers' if year <= 2016 else 'Los Angeles Chargers'
    if abbr in ['OAK', 'LV']:
        return 'Oakland Raiders' if year <= 2019 else 'Las Vegas Raiders'
    return mapping.get(abbr)


In [ ]:
# Create unique coach ID and experience counter
df['CoachID'] = df['CoachName'] + "_" + df['DOB'].astype(str)
df = df.sort_values(['CoachID', 'Year'])
df['Experience'] = df.groupby('CoachID').cumcount()

# Filter to 2003+ and standardize team names
df_filtered = df[df['Year'] >= 2003].copy()
df_filtered['FullTeamName'] = df_filtered.apply(get_historical_name, axis=1)

# Merge coaching data with team stats
merged_df = pd.merge(
    df_filtered, df2,
    left_on=['Year', 'FullTeamName'], right_on=['year', 'team'],
    how='left'
).drop(columns=['year', 'team'])

print(f"Missing FullTeamNames: {merged_df['FullTeamName'].isnull().sum()}")
print(f"Merged dataset: {merged_df.shape}")


In [ ]:
# Convert ties to half-wins/half-losses and drop the ties column
has_tie = merged_df['ties'] > 0
merged_df.loc[has_tie, 'wins'] += merged_df['ties'] * 0.5
merged_df.loc[has_tie, 'losses'] += merged_df['ties'] * 0.5
merged_df = merged_df.drop(columns=['ties'])

# Drop rows with missing coach names
merged_df['CoachName'] = merged_df['CoachName'].replace('', pd.NA)
merged_df = merged_df.dropna(subset=['CoachName'])

print(f"Dataset size after cleaning: {len(merged_df)} rows")
print(f"Unique coaches: {merged_df['CoachID'].nunique()}")


## 4. Feature Engineering

### 4a. Role Hierarchy, Movement Labels & Race Encoding

In [ ]:
# Define 3-tier coaching hierarchy
original_hierarchy = {
    'HC': 3,
    'OC': 2, 'DC': 2, 'ST': 2,
    'QB': 1,
    'DL': 1, 'WR': 1, 'DB': 1, 'RB': 1,
    'LB': 1, 'OL': 1, 'TE': 1
}

merged_df['Tier'] = merged_df['Role'].map(original_hierarchy)

# Calculate forward-looking movement labels
merged_df = merged_df.sort_values(['CoachID', 'Year'])
merged_df['Next_Tier'] = merged_df.groupby('CoachID')['Tier'].shift(-1)
merged_df['Next_Year'] = merged_df.groupby('CoachID')['Year'].shift(-1)

def calculate_forward_movement(row):
    if pd.isna(row['Next_Tier']) or (row['Next_Year'] - row['Year'] > 1):
        return 'Exit/Gap'
    if row['Next_Tier'] > row['Tier']:
        return 'Promotion'
    elif row['Next_Tier'] < row['Tier']:
        return 'Demotion'
    return 'Sideways'

merged_df['Movement'] = merged_df.apply(calculate_forward_movement, axis=1)

print("Original Movement distribution:")
print(merged_df['Movement'].value_counts())


In [ ]:
# 2025 demographics snapshot (before filtering)
df_2025 = merged_df[merged_df['Year'] == 2025]
label_map = {'W': 'White', 'B': 'Black', 'M': 'Mixed/Other', '0': 'Unknown'}
print("--- 2025 Coaching Demographics ---")
for race_val, count in df_2025['Race'].value_counts().items():
    pct = count / len(df_2025) * 100
    print(f"{label_map.get(race_val, race_val)}: {count} ({pct:.1f}%)")

# Keep only White (W) and Black (B) coaches, encode as 0/1
merged_df = merged_df[merged_df['Race'].isin(['W', 'B'])].copy()
merged_df['Race'] = merged_df['Race'].map({'W': 0, 'B': 1})

print(f"\nAfter race filtering: {len(merged_df)} rows")
print(merged_df['Race'].value_counts().rename({0: 'White', 1: 'Black'}))


### 4b. Statistical Positionality Score

In [ ]:
#Define relevant statistics for each position
relevant_stats = {
    'HC': ['win_loss_perc', 'mov', 'points_diff', 'wins'],
    'OC': ['points', 'score_pct', 'exp_pts_tot'],
    'DC': ['points_opp', 'turnover_pct'],
    'QB': ['pass_yds', 'pass_td', 'pass_cmp_perc', 'pass_int', 'pass_fd'],
    'RB': ['rush_yds', 'rush_td', 'rush_yds_per_att', 'rush_fd'],
    'WR': ['pass_yds', 'pass_td', 'pass_fd'],
    'TE': ['pass_yds', 'pass_td', 'pass_fd'],
    'OL': ['rush_yds_per_att', 'rush_att', 'penalties'],
    'DL': ['points_opp'], 'LB': ['points_opp'], 
    'DB': ['points_opp', 'pass_int'],
}

def calculate_statistical_positionality(df):
    df['statistical_positionality'] = 0.0
    inverse_stats = {'points_opp', 'pass_int', 'penalties', 'penalties_yds'}
    
    for (year, role), group in df.groupby(['Year', 'Role']):
        if role not in relevant_stats:
            continue
        stats_to_check = [s for s in relevant_stats[role] if s in group.columns]
        if not stats_to_check:
            continue
        
        percentile_ranks = []
        for stat in stats_to_check:
            ascending = stat not in inverse_stats
            percentile_ranks.append(group[stat].rank(pct=True, ascending=ascending))
        
        avg_percentile = pd.concat(percentile_ranks, axis=1).mean(axis=1)
        df.loc[avg_percentile.index, 'statistical_positionality'] = avg_percentile
    return df

merged_df = calculate_statistical_positionality(merged_df)
print(merged_df[['CoachName', 'Year', 'Role', 'statistical_positionality']].sample(5))


### 4c. League-Wide Churn Rate Feature

In [ ]:
def calculate_seasonal_churn(df):
    churn_data = []
    df_sorted = df.sort_values(['FullTeamName', 'Role', 'Year'])
    years = sorted(df_sorted['Year'].unique())
    
    for i in range(len(years) - 1):
        current = df_sorted[df_sorted['Year'] == years[i]][['CoachID', 'FullTeamName', 'Role']]
        next_yr = df_sorted[df_sorted['Year'] == years[i+1]][['CoachID', 'FullTeamName', 'Role']]
        returned = pd.merge(current, next_yr, on=['CoachID', 'FullTeamName', 'Role'])
        
        total = len(current)
        persistence = len(returned) / total if total > 0 else 0
        churn_data.append({
            'Year': years[i], 'Next_Year': years[i+1],
            'Churn_Rate': 1 - persistence, 'Total_Coaches': total, 'Returned': len(returned)
        })
    return pd.DataFrame(churn_data)

churn_df = calculate_seasonal_churn(merged_df)

# Map previous year's churn as a feature
churn_lookup = churn_df[['Next_Year', 'Churn_Rate']].copy()
churn_lookup['Target_Year'] = churn_lookup['Next_Year'] + 1
churn_lookup = churn_lookup.rename(columns={'Churn_Rate': 'prev_league_churn'})

merged_df = pd.merge(
    merged_df, churn_lookup[['Target_Year', 'prev_league_churn']],
    left_on='Year', right_on='Target_Year', how='left'
).drop(columns=['Target_Year'])
merged_df['prev_league_churn'] = merged_df['prev_league_churn'].fillna(merged_df['prev_league_churn'].mean())

# Visualization
plt.figure(figsize=(14, 6))
sns.lineplot(data=churn_df, x='Year', y='Churn_Rate', marker='o', color='#e74c3c', linewidth=2.5)
mean_churn = churn_df['Churn_Rate'].mean()
plt.axhline(mean_churn, color='black', linestyle=':', label=f'Average: {mean_churn:.2%}')
plt.title('NFL Coaching Churn Rate (2003-2025)', fontsize=14, fontweight='bold')
plt.ylabel('Churn Rate'); plt.xlabel('Season'); plt.ylim(0, 1)
plt.legend(); plt.grid(True, linestyle='--', alpha=0.6); plt.tight_layout(); plt.show()


## 5. Exploratory Data Analysis

In [ ]:
merged_df.describe()


In [ ]:
merged_df.info()


### 5a. Feature Selection (F-Scores)

In [ ]:
numerical_features = list(merged_df.select_dtypes(include=['float64', 'int64', 'int16']).columns)

analysis_df = merged_df[merged_df['Movement'] != 'Exit/Gap'].copy()
f_results = {}
for col in numerical_features:
    temp = analysis_df[[col, 'Movement']].dropna()
    if len(temp) > 1 and temp['Movement'].nunique() > 1:
        f_val, p_val = f_classif(temp[[col]], temp['Movement'])
        f_results[col] = f_val[0]

print("Ranked F-Scores for Numerical Features:")
print(pd.Series(f_results).sort_values(ascending=False))


### 5b. Movement by Role & Tier Visualizations

In [ ]:
# Movement distribution by role
for role in merged_df['Role'].unique():
    role_df = merged_df[merged_df['Role'] == role]
    if role_df.empty:
        continue
    plt.figure(figsize=(8, 4))
    sns.countplot(data=role_df, x='Movement',
                  order=['Promotion', 'Sideways', 'Demotion', 'Exit/Gap'], palette='magma')
    plt.title(f'Movement Outcomes for "{role}"', fontsize=13)
    plt.xlabel('Movement'); plt.ylabel('Count')
    plt.grid(axis='y', linestyle='--', alpha=0.6); plt.tight_layout(); plt.show()


In [ ]:
# Role distribution by movement category
for category in merged_df['Movement'].unique():
    cat_df = merged_df[merged_df['Movement'] == category]
    if cat_df.empty:
        continue
    plt.figure(figsize=(10, 5))
    sns.countplot(data=cat_df, x='Role', order=cat_df['Role'].value_counts().index, palette='viridis')
    plt.title(f'Role Distribution for "{category}" Movement', fontsize=13)
    plt.xlabel('Role'); plt.ylabel('Count'); plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7); plt.tight_layout(); plt.show()


In [ ]:
# Movement counts by tier
rank = {'Promotion': 3, 'Sideways': 2, 'Demotion': 1, 'Exit/Gap': 0}
merged_df['Movement_Num'] = merged_df['Movement'].map(rank)

plt.figure(figsize=(12, 7))
sns.countplot(data=merged_df, x='Tier', hue='Movement_Num', palette='viridis')
plt.title('Career Movements by Coaching Tier', fontsize=14, fontweight='bold')
plt.xlabel('Tier (1=Support, 5=HC)'); plt.ylabel('Count')
plt.legend(title='Movement', labels=['0: Exit/Gap', '1: Demotion', '2: Sideways', '3: Promotion'])
plt.grid(axis='y', linestyle='--', alpha=0.6); plt.tight_layout(); plt.show()


## 6. Preprocessing & Model Training



In [ ]:
# Define features and target
features = ['Race', 'Tier', 'wins', 'points_diff', 'score_pct', 'mov',
            'statistical_positionality', 'Experience', 'prev_league_churn']

X = merged_df[features]
y = merged_df['Movement']
groups = merged_df['CoachID']   # one coach has many seasons; keep each coach in ONE split

# --- Group-aware train/test split (fixes the original random-split leakage) ---
# Labels are forward-looking and the same coach recurs across seasons, so a random
# split scatters one coach across train AND test. We split on CoachID so that no
# coach appears in both sets.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]   # used for group-aware cross-validation below

# Cross-validation strategy reused everywhere below: respects groups AND stratifies by class
cv_strategy = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# Sanity check: zero coach overlap between train and test
overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"Coaches in both train & test (should be 0): {len(overlap)}")
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")
print(f"\nMissing values in training set:\n{X_train.isnull().sum()}")
print(f"\nClass distribution (train):\n{y_train.value_counts()}")


### 6a. Define the Preprocessor 




In [ ]:
# All features in our set are numerical (Race and Tier are encoded as ints)
numerical_features_list = features  # all features are numeric

# Preprocessing pipeline for numerical features:
# Step 1: Impute missing values with the MEDIAN (robust to outliers)
# Step 2: Standardize features to zero mean and unit variance
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# ColumnTransformer applies the pipeline to the correct columns
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features_list)
])

print("Preprocessor defined:")
print(preprocessor)


### 6b. Baseline Model Comparison

In [ ]:
# Define models — each wrapped in a Pipeline with the shared preprocessor
model_dict = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Naive Bayes": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "LDA": LinearDiscriminantAnalysis()
}

# Helper: build a full pipeline for each model
def make_pipeline(model):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

# Train, evaluate, and audit each model
for name, model in model_dict.items():
    pipe = make_pipeline(model)
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    
    # Classification report
    print(f"\n{'='*25} {name.upper()} {'='*25}")
    print(classification_report(y_test, preds))
    
    # Cross-validation score
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv_strategy,
                                groups=groups_train, scoring='f1_macro')
    print(f"CV Macro-F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    
    # Confusion matrix
    disp = ConfusionMatrixDisplay.from_predictions(
        y_test, preds, display_labels=pipe.classes_, cmap='Blues', xticks_rotation='vertical')
    plt.title(f"{name} Confusion Matrix"); plt.show()
    
    # Bias audit
    df_audit = X_test.copy()
    df_audit['Actual'] = y_test
    df_audit['Predicted'] = preds
    df_audit['Correct'] = df_audit['Actual'] == df_audit['Predicted']
    
    print(f"Overall Accuracy: {df_audit['Correct'].mean():.2%}")
    print("\nAccuracy by Race (0=White, 1=Black):")
    print(df_audit.groupby('Race')['Correct'].mean().round(4))
    
    mis = df_audit[~df_audit['Correct']].copy()
    mis['Actual_Rank'] = mis['Actual'].map(rank)
    mis['Predicted_Rank'] = mis['Predicted'].map(rank)
    mis['Error_Type'] = mis.apply(
        lambda r: 'Optimistic' if r['Predicted_Rank'] > r['Actual_Rank'] else 'Pessimistic', axis=1)
    
    print(f"\n{name} Bias Summary (% of errors):")
    bias_table = mis.groupby(['Race', 'Error_Type']).size().unstack(fill_value=0)
    display(bias_table.div(bias_table.sum(axis=1), axis=0).mul(100).round(2))
    
    print("\nTop 3 Mispredictions for Black Coaches (Race=1):")
    top3 = mis[mis['Race'] == 1].groupby(['Actual', 'Predicted']).size().sort_values(ascending=False).head(3)
    print(top3)


# 6c. Addressing Imbalance

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline

balanced_trees = {
    "Random Forest (Balanced)": RandomForestClassifier(random_state=42, class_weight='balanced'),
    "Decision Tree (Balanced)": DecisionTreeClassifier(random_state=42, class_weight='balanced')
}

print("--- SCENARIO 1: CLASS WEIGHT BALANCING AUDIT ---")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for i, (name, model) in enumerate(balanced_trees.items()):
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    
    # 1. Full Classification Report
    print(f"\n{'='*20} {name.upper()} {'='*20}")
    print(classification_report(y_test, preds))
    
    # 2. Accuracy by Race Audit
    df_audit = X_test.copy()
    df_audit['Correct'] = (preds == y_test.values)
    print(f"Overall Accuracy: {df_audit['Correct'].mean():.2%}")
    print("Accuracy by Race (0=White, 1=Black):")
    print(df_audit.groupby('Race')['Correct'].mean().round(4))
    
    ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[i], cmap='Blues', xticks_rotation='vertical')
    axes[i].set_title(name)

plt.tight_layout()
plt.show()

# Pre-transform for SMOTE
X_train_pre = preprocessor.fit_transform(X_train)
X_test_pre = preprocessor.transform(X_test)

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_pre, y_train)

smote_models = {
    "Random Forest (SMOTE)": RandomForestClassifier(random_state=42),
    "Decision Tree (SMOTE)": DecisionTreeClassifier(random_state=42),
    "Naive Bayes (SMOTE)": GaussianNB(),
    "KNN (SMOTE)": KNeighborsClassifier(n_neighbors=5),
    "LDA (SMOTE)": LinearDiscriminantAnalysis()
}

print("\n--- SCENARIO 2: SMOTE OVERSAMPLING AUDIT ---")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (name, model) in enumerate(smote_models.items()):
    model.fit(X_resampled, y_resampled)
    preds = model.predict(X_test_pre)
    
    # 1. Full Classification Report
    print(f"\n{'='*20} {name.upper()} {'='*20}")
    print(classification_report(y_test, preds))
    
    # 2. Accuracy by Race Audit
    df_audit_smote = X_test.copy()
    df_audit_smote['Correct'] = (preds == y_test.values)
    print(f"Overall Accuracy: {df_audit_smote['Correct'].mean():.2%}")
    print("Accuracy by Race (0=White, 1=Black):")
    print(df_audit_smote.groupby('Race')['Correct'].mean().round(4))
    
    ConfusionMatrixDisplay.from_predictions(y_test, preds, display_labels=pipe.classes_, 
                                           ax=axes[i], cmap='Greens', xticks_rotation='vertical')
    axes[i].set_title(name)

fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()

In [ ]:
# 1. Update the hierarchy mapping (drop Special Teams, move to a 5-tier scheme)
merged_df = merged_df[merged_df['Role'] != 'ST'].copy()
final_hierarchy = {
    'HC': 5,
    'OC': 4, 'DC': 4,
    'QB': 3,
    'DL': 2, 'WR': 2, 'DB': 2, 'RB': 2,
    'LB': 1, 'OL': 1, 'TE': 1
}

# 2. Re-map Tiers and recompute forward-looking Movement labels
merged_df['Tier'] = merged_df['Role'].map(final_hierarchy)
merged_df = merged_df.sort_values(['CoachID', 'Year'])
merged_df['Next_Tier'] = merged_df.groupby('CoachID')['Tier'].shift(-1)
merged_df['Next_Year'] = merged_df.groupby('CoachID')['Year'].shift(-1)  # refresh: was stale after re-sort
merged_df['Movement'] = merged_df.apply(calculate_forward_movement, axis=1)

# 3. Refresh features/target and re-do the GROUP-AWARE split (no coach in both sets)
X = merged_df[features]
y = merged_df['Movement']
groups = merged_df['CoachID']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print("Hierarchy updated to 5 Tiers for Final Evaluation.")
print(f"Coaches in both train & test (should be 0): {len(overlap)}")
print("New Class Distribution (Train):")
print(y_train.value_counts())


## 7. Hyperparameter Tuning (GridSearchCV)

In [ ]:
# Random Forest GridSearch — using pipeline so preprocessing is inside CV
rf_pipeline = make_pipeline(RandomForestClassifier(random_state=42))

rf_param_grid = {
    'classifier__n_estimators': np.arange(50,500,50),
    'classifier__max_depth': np.arange(5,50,5),
    'classifier__min_samples_split': np.arange(2,6),
    'classifier__min_samples_leaf': np.arange(2,10),
    'classifier__class_weight': ['balanced', 'balanced_subsample']
}

print("Running Random Forest GridSearch...")
rf_grid = GridSearchCV(rf_pipeline, rf_param_grid, cv=cv_strategy, n_jobs=-1, scoring='f1_macro')
rf_grid.fit(X_train, y_train, groups=groups_train)

print(f"\nBest Parameters: {rf_grid.best_params_}")
print(f"Best CV Macro-F1: {rf_grid.best_score_:.4f}")


In [ ]:
# Decision Tree GridSearch
dt_pipeline = make_pipeline(DecisionTreeClassifier(random_state=42, class_weight='balanced'))

dt_param_grid = {
    'classifier__max_depth': np.arange(5, 30, 5),
}

print("Running Decision Tree GridSearch...")
dt_grid = GridSearchCV(dt_pipeline, dt_param_grid, cv=cv_strategy, n_jobs=-1, scoring='f1_macro')
dt_grid.fit(X_train, y_train, groups=groups_train)

print(f"\nBest Parameters: {dt_grid.best_params_}")
print(f"Best CV Macro-F1: {dt_grid.best_score_:.4f}")


## 8. Final Model Evaluation

In [ ]:
# Use the best model from grid search
best_model = rf_grid.best_estimator_
final_preds = best_model.predict(X_test)

print("Final Optimized Model Performance (Test Set):")
print(classification_report(y_test, final_preds))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, final_preds, display_labels=best_model.classes_, cmap='Blues', xticks_rotation='vertical')
plt.title("Final Random Forest — Confusion Matrix"); plt.show()

# Final bias audit
df_final = X_test.copy()
df_final['Actual'] = y_test
df_final['Predicted'] = final_preds
df_final['Correct'] = df_final['Actual'] == df_final['Predicted']

print("\nAccuracy by Race (0=White, 1=Black):")
print(df_final.groupby('Race')['Correct'].mean().round(4))

mis = df_final[~df_final['Correct']].copy()
mis['Actual_Rank'] = mis['Actual'].map(rank)
mis['Predicted_Rank'] = mis['Predicted'].map(rank)
mis['Error_Type'] = mis.apply(
    lambda r: 'Optimistic' if r['Predicted_Rank'] > r['Actual_Rank'] else 'Pessimistic', axis=1)

print("\nFinal Bias Summary (% of errors per group):")
bias_table = mis.groupby(['Race', 'Error_Type']).size().unstack(fill_value=0)
display(bias_table.div(bias_table.sum(axis=1), axis=0).mul(100).round(2))


## 9. Feature Importance Analysis

In [ ]:
# Extract the classifier from the pipeline
final_rf = best_model.named_steps['classifier']

# MDI importance
mdi = final_rf.feature_importances_

# Permutation importance (on preprocessed test data)
X_test_processed = best_model.named_steps['preprocessor'].transform(X_test)
perm = permutation_importance(final_rf, X_test_processed, y_test,
                               n_repeats=10, random_state=42, n_jobs=-1, scoring='f1_macro')

importance_df = pd.DataFrame({
    'Feature': features,
    'MDI_Importance': mdi,
    'Permutation_Importance': perm.importances_mean,
    'Perm_Std': perm.importances_std
}).sort_values('Permutation_Importance', ascending=False)

print("--- Feature Importance Comparison ---")
display(importance_df.reset_index(drop=True))
